In [17]:
import requests
import pandas as pd
import isodate

#### Trouver l'ID de la playlist

In [18]:
# url = "https://youtube.googleapis.com/youtube/v3/channels?part=contentDetails&forHandle=AlexTheAnalyst&key=[]"
# response = requests.get(url)
# res = response.json()
# print(res)

In [19]:
# playlist_id = res["items"][0]["contentDetails"]["relatedPlaylists"]["uploads"]
# playlist_id

#### Récuperer les IDs des vidéos 

In [20]:
# url1 = 'https://youtube.googleapis.com/youtube/v3/playlistItems?part=contentDetails&playlistId=UU7cs8q-gJRlGwj4A8OmCmXg&key=[]'
# response1 = requests.get(url1)
# response1.json()

## 1. Récupération des identifiants :

#### Fonction pour récupérer l'id de la playlist :

In [21]:
def get_uploads_playlist_id(channel_id, api_key):
    
    url = "https://www.googleapis.com/youtube/v3/channels"
    
    params = {
        "part": "contentDetails",   
        "id": channel_id,          
        "key": api_key              
    }
    
    response = requests.get(url, params=params).json()
    print(response)
    
    playlist_id = response["items"][0]["contentDetails"]["relatedPlaylists"]["uploads"]
    
    return playlist_id

#### Fonction pour récupérer les ids des vidéos :

In [22]:
def get_video_ids(playlist_id, api_key):
    
    video_ids = []           
    next_page_token = None   
    
    while True:   
        
        params = {
            "part": "contentDetails",
            "playlistId": playlist_id,  
            "maxResults": 50,           
            "pageToken": next_page_token, 
            "key": api_key
        }
        
        response = requests.get(
            "https://www.googleapis.com/youtube/v3/playlistItems",
            params=params
        ).json()
        
        for item in response["items"]:
            video_ids.append(item["contentDetails"]["videoId"])
        
        next_page_token = response.get("nextPageToken")
        
        if not next_page_token:
            break   
    
    return video_ids   

#### Fonction pour récupérer les détails des vidéos :

In [23]:
def get_video_details(video_ids, api_key):
    
    all_videos = []   
    
    for i in range(0, len(video_ids), 50):
        
        batch = video_ids[i:i+50]   
        
        params = {
            "part": "snippet,statistics,contentDetails",
            
            "id": ",".join(batch),   
            
            "key": api_key
        }
        
        response = requests.get(
            "https://www.googleapis.com/youtube/v3/videos",
            params=params
        ).json()
        
        all_videos.extend(response["items"])
    
    return all_videos

## 2. Sauvegarde & transformation des données :

#### Fonction pour sauvegarder les données brutes :

In [24]:
import json
import os

def save_raw(data, filename="videos_raw.json"):

    os.makedirs("data/raw", exist_ok=True)

    filepath = f"data/raw/{filename}"

    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=4)

#### Fonction pour sauvegarder les données brutes :

In [25]:
def load_raw(filename="videos_raw.json"):

    filepath = f"data/raw/{filename}"

    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)

    return data

In [26]:
def transform(raw_videos):
    
    records = []   
    
    for v in raw_videos:  
        
        snippet = v.get("snippet", {})
        stats   = v.get("statistics", {})
        content = v.get("contentDetails", {})
        
        records.append({
            "video_id"    : v["id"],
            "title"       : snippet.get("title"),
            "published_at": pd.to_datetime(snippet.get("publishedAt")),
            "duration_sec": int(isodate.parse_duration(
                                content.get("duration", "PT0S")
                            ).total_seconds()),
            "views"    : int(stats.get("viewCount", 0)),
            "likes"    : int(stats.get("likeCount", 0)),
            "comments" : int(stats.get("commentCount", 0)),
            "tags"         : ",".join(snippet.get("tags", [])),
            "category_id"  : snippet.get("categoryId"),
            "definition"   : content.get("definition"),# "hd" ou "sd"
            "has_captions" : content.get("caption") == "true",
        }) 
    df = pd.DataFrame(records)   
    df.dropna(subset=["title"], inplace=True)
    return df 

#### Fonction pour sauvegarder le fichier csv final :

In [27]:
def load(df, filename="youtube_data.csv"):

    os.makedirs("data/processed", exist_ok=True)

    filepath = f"data/processed/{filename}"

    df.to_csv(filepath, index=False, encoding="utf-8-sig")

## 3. Pipeline complet :

In [ ]:
from dotenv import load_dotenv

load_dotenv()
API_KEY    = os.getenv("YOUTUBE_API_KEY")
CHANNEL_ID = os.getenv("CHANNEL_ID")


playlist_id = get_uploads_playlist_id(CHANNEL_ID, API_KEY)
print(f"   Playlist ID trouvé : {playlist_id}")

video_ids = get_video_ids(playlist_id, API_KEY)
print(f"   {len(video_ids)} vidéos trouvées")

raw_data = get_video_details(video_ids, API_KEY)

save_raw(raw_data)


raw_data_from_file = load_raw()

df = transform(raw_data_from_file)

load(df)

{'error': {'code': 403, 'message': "Method doesn't allow unregistered callers (callers without established identity). Please use API Key or other form of API consumer identity to call this API.", 'errors': [{'message': "Method doesn't allow unregistered callers (callers without established identity). Please use API Key or other form of API consumer identity to call this API.", 'domain': 'global', 'reason': 'forbidden'}], 'status': 'PERMISSION_DENIED'}}


KeyError: 'items'